# 03d · BiLSTM v2: rolling yearly validation (+ Reddit LLM labels when available)

Changes vs 03b (same tokenizer, word2vec and architecture, so the comparison is clean):

1. **2021 is now in training.** The number of epochs is chosen by **rolling (expanding-window) yearly validation**:

   | fold | train (FNSPID) | validate |
   |---|---|---|
   | 1 | 2009–2018 | 2019 |
   | 2 | 2009–2019 | 2020 |
   | 3 | 2009–2020 | 2021 |

   Every fold also trains on the Twitter train split (and Reddit train, if available). The epoch with the best average validation macro-F1 is used for the final model, trained on **all FNSPID 2009–2021**. **2022–23 stays the untouched test.**
2. **Reddit labels from 03c** (Qwen, `llm_labels_reddit.csv`) are added automatically if that file exists: Reddit train days for training, val days for epoch selection, test days for testing. If the file is missing (no GPU yet for 03c), the notebook runs without them; re-run it after 03c.

**Outputs (DataSource/):** `bilstm_{TAG}_metrics.csv`, `_cv.csv`, `_scores_fnspid.csv`, `_scores_reddit.csv`, `_sentiment.pt`, where TAG = `v2_rolling` (run without Reddit labels) or `v2_reddit` (run after 03c). The 03b (v1) files are not touched.

In [2]:
# 0. Setup
!pip -q install "gensim>=4.4"
from google.colab import drive
drive.mount('/content/drive')
import os, glob, re, time, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix
from scipy import stats

def find_project_dir(name="dl-quant_project"):
    cands = [f"/content/drive/MyDrive/Colab Notebooks/{name}", f"/content/drive/MyDrive/{name}",
             *glob.glob(f"/content/drive/MyDrive/*/{name}"), *glob.glob(f"/content/drive/Shareddrives/*/{name}")]
    for p in cands:
        if os.path.isdir(p): return p
    raise FileNotFoundError("Add a shortcut to the shared 'dl-quant_project' folder in My Drive (see 00_START_HERE).")
DATA_DIR = os.path.join(find_project_dir(), "DataSource")
SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count())
LABELS = ["bearish", "neutral", "bullish"]; L2I = {l: i for i, l in enumerate(LABELS)}
REDDIT_LABELS = os.path.join(DATA_DIR, "llm_labels_reddit.csv")
USE_REDDIT = os.path.exists(REDDIT_LABELS)
TAG = "v2_reddit" if USE_REDDIT else "v2_rolling"      # output-file tag: rolling only, or rolling + Reddit labels
pd.set_option("display.width", 200)
print("device:", DEVICE, "| Reddit LLM labels available:", USE_REDDIT)

Mounted at /content/drive
device: cuda | Reddit LLM labels available: True


## 1. Data

In [3]:
from datasets import load_dataset
L = pd.read_csv(os.path.join(DATA_DIR, "llm_labels_fnspid.csv"))
L["year"] = pd.to_datetime(L["date"]).dt.year
L["text"] = L["input_text"].fillna(L["headline"]).astype(str)
tw = load_dataset("zeroshot/twitter-financial-news-sentiment")
TW_MAP = {0: "bearish", 1: "bullish", 2: "neutral"}
tw_train = tw["train"].to_pandas(); tw_train["label_name"] = tw_train.label.map(TW_MAP)
tw_test = tw["validation"].to_pandas(); tw_test["label_name"] = tw_test.label.map(TW_MAP)

def onehot(names):
    Q = np.zeros((len(names), 3), dtype=np.float32); Q[np.arange(len(names)), [L2I[n] for n in names]] = 1.0; return Q

blocks = [pd.DataFrame({"text": L.text, "source": "fnspid", "year": L.year, "rsplit": "", "y": L.llm_label.map(L2I),
                        "w": L.llm_confidence, "q0": L.p_bear, "q1": L.p_neu, "q2": L.p_bull}),
          pd.DataFrame({"text": tw_train.text, "source": "twitter", "year": 0, "rsplit": "", "y": tw_train.label_name.map(L2I), "w": 1.0})
            .join(pd.DataFrame(onehot(tw_train.label_name), columns=["q0", "q1", "q2"]))]
if USE_REDDIT:
    RL = pd.read_csv(REDDIT_LABELS)
    blocks.append(pd.DataFrame({"text": RL.text, "source": "reddit", "year": 2021, "rsplit": RL.split, "y": RL.llm_label.map(L2I),
                                "w": RL.llm_confidence, "q0": RL.p_bear, "q1": RL.p_neu, "q2": RL.p_bull}))
data = pd.concat(blocks, ignore_index=True)
print(data.groupby(["source", "rsplit"]).size())
print("\nFNSPID by year:", L.year.value_counts().sort_index().to_dict())

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

source   rsplit
fnspid             8784
reddit   test       867
         train     4255
         val        878
twitter            9543
dtype: int64

FNSPID by year: {2009: 9, 2010: 90, 2011: 130, 2012: 206, 2013: 210, 2014: 263, 2015: 408, 2016: 470, 2017: 539, 2018: 784, 2019: 746, 2020: 742, 2021: 2529, 2022: 1034, 2023: 624}


## 2. Tokenizer, word2vec, vocabulary (identical to 03b)

In [4]:
URL = re.compile(r"https?://\S+|www\.\S+")
TOK = re.compile(r"\$[a-z]+|[a-z]+(?:'[a-z]+)?|\d+(?:[.,]\d+)?%?|[%!?]")
def tokenize(s):
    s = URL.sub(" ", str(s).lower()); out = []
    for t in TOK.findall(s):
        if t.startswith("$"): out.append("<cashtag>")
        elif t[0].isdigit(): out.append("<pct>" if t.endswith("%") else "<num>")
        else: out.append(t)
    return out

parts = []
for f in sorted(glob.glob(os.path.join(DATA_DIR, "wallstreetbets_*.csv"))):
    d = pd.read_csv(f, usecols=["id", "body", "created_utc", "score"]); d["ticker"] = os.path.basename(f).split("_")[1]; parts.append(d)
R = pd.concat(parts, ignore_index=True).drop_duplicates(["ticker", "id"])
R = R[R.body.notna() & ~R.body.isin(["[deleted]", "[removed]"])].reset_index(drop=True)
clean = pd.read_csv(os.path.join(DATA_DIR, "fnspid_meme_clean.csv"))
clean = clean[pd.to_datetime(clean.date).dt.year <= 2021]
corpus = [tokenize(s) for s in (list(clean.Article_title.fillna("") + " . " + clean.text.fillna("").str.slice(0, 5000))
                                + list(tw_train.text) + list(R.body.sample(min(len(R), 300_000), random_state=SEED)))]
from gensim.models import Word2Vec
t0 = time.time()
w2v = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=3, sg=1, negative=5, epochs=5, workers=os.cpu_count(), seed=SEED)
print(f"word2vec: {sum(map(len, corpus)):,} tokens, vocab {len(w2v.wv):,}, {time.time()-t0:.0f}s")

MAX_VOCAB, MAX_LEN, EMB = 40_000, 64, 100
itos = ["<pad>", "<unk>"] + w2v.wv.index_to_key[:MAX_VOCAB]; stoi = {w: i for i, w in enumerate(itos)}
emb_matrix = np.zeros((len(itos), EMB), dtype=np.float32); emb_matrix[1] = np.random.default_rng(SEED).normal(0, 0.1, EMB)
for w, i in stoi.items():
    if i >= 2: emb_matrix[i] = w2v.wv[w]

def encode(texts, voc=None, max_len=MAX_LEN):
    voc = voc or stoi
    X = np.zeros((len(texts), max_len), dtype=np.int64); lens = np.ones(len(texts), dtype=np.int64)
    for r, s in enumerate(texts):
        ids = [voc.get(t, 1) for t in tokenize(s)][:max_len] or [1]; X[r, :len(ids)] = ids; lens[r] = len(ids)
    return torch.from_numpy(X), torch.from_numpy(lens)
X_all, len_all = encode(data.text.tolist())
print("encoded", len(data))

word2vec: 10,284,948 tokens, vocab 30,922, 226s
encoded 24327


## 3. Model and training loop

In [5]:
class BiLSTMClf(nn.Module):
    def __init__(self, emb, hidden=64, n_cls=3, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.tensor(emb), freeze=False, padding_idx=0)
        self.emb_drop = nn.Dropout(0.2)
        self.lstm = nn.LSTM(emb.shape[1], hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop); self.fc = nn.Linear(4 * hidden, n_cls)
    def forward(self, x, lens):
        e = self.emb_drop(self.emb(x))
        out, _ = self.lstm(nn.utils.rnn.pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False))
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        mask = (x != 0).unsqueeze(-1)
        mean = (out * mask).sum(1) / mask.sum(1).clamp(min=1); mx = out.masked_fill(~mask, -1e4).max(1).values
        return self.fc(self.drop(torch.cat([mean, mx], 1)))

Q_t = torch.tensor(data[["q0", "q1", "q2"]].to_numpy(np.float32)); W_t = torch.tensor(data.w.to_numpy(np.float32))
y_all = data.y.to_numpy(); Y_t = torch.tensor(y_all)

@torch.no_grad()
def predict_proba(model, X, lens, bs=1024):
    model.eval()
    return torch.cat([F.softmax(model(X[i:i+bs].to(DEVICE), lens[i:i+bs].to(DEVICE)), -1).cpu() for i in range(0, len(X), bs)]).numpy()

def train(tr_idx, epochs, seed, evals=None, bs=64, lr=2e-3, emb=None):
    # evals: {name: index array}; returns model and per-epoch macro-F1 on each eval set
    set_seed(seed)
    model = BiLSTMClf(emb_matrix if emb is None else emb).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    counts = np.bincount(y_all[tr_idx], minlength=3); cw = torch.tensor(len(tr_idx) / (3 * counts), dtype=torch.float32).to(DEVICE)
    hist = []
    for ep in range(1, epochs + 1):
        model.train(); perm = np.random.permutation(tr_idx); t0 = time.time()
        for i in range(0, len(perm), bs):
            b = torch.from_numpy(perm[i:i+bs])
            logits = model(X_all[b].to(DEVICE), len_all[b].to(DEVICE))
            loss = (-(Q_t[b].to(DEVICE) * F.log_softmax(logits, -1)).sum(1) * cw[Y_t[b].to(DEVICE)] * W_t[b].to(DEVICE)).mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        row = {"epoch": ep, "sec": time.time() - t0}
        for name, idx in (evals or {}).items():
            row[name] = f1_score(y_all[idx], predict_proba(model, X_all[idx], len_all[idx]).argmax(1), average="macro")
        hist.append(row)
        if evals: print("   ", {k: round(v, 3) for k, v in row.items()})
    return model, pd.DataFrame(hist)

## 4. Rolling yearly validation → choose the number of epochs

In [6]:
EPOCHS_CV, FOLD_YEARS = 12, [2019, 2020, 2021]
src, yr, rs = data.source.to_numpy(), data.year.to_numpy(), data.rsplit.to_numpy()
base_train = np.where((src == "twitter") | ((src == "reddit") & (rs == "train")))[0]
reddit_val = np.where((src == "reddit") & (rs == "val"))[0]
cv = []
for k in FOLD_YEARS:
    tr_idx = np.concatenate([np.where((src == "fnspid") & (yr < k))[0], base_train])
    evals = {"fnspid_val": np.where((src == "fnspid") & (yr == k))[0]}
    if USE_REDDIT: evals["reddit_val"] = reddit_val
    print(f"fold: train FNSPID < {k} (+Twitter{' +Reddit' if USE_REDDIT else ''}), n={len(tr_idx):,}; validate on {k} (n={len(evals['fnspid_val'])})")
    _, h = train(tr_idx, EPOCHS_CV, SEED, evals)
    cv.append(h.assign(val_year=k))
CV = pd.concat(cv, ignore_index=True)
score_cols = ["fnspid_val"] + (["reddit_val"] if USE_REDDIT else [])
by_ep = CV.groupby("epoch")[score_cols].mean()
by_ep["score"] = by_ep[score_cols].mean(1)
BEST_EPOCHS = int(by_ep.score.idxmax())
print("\nmean validation macro-F1 by epoch:\n", by_ep.round(3).to_string())
print("\nper fold:\n", CV.pivot_table(index="epoch", columns="val_year", values="fnspid_val").round(3).to_string())
print(f"\n→ chosen number of epochs: {BEST_EPOCHS}")
CV.to_csv(os.path.join(DATA_DIR, f"bilstm_{TAG}_cv.csv"), index=False)

fold: train FNSPID < 2019 (+Twitter +Reddit), n=16,907; validate on 2019 (n=746)
    {'epoch': 1, 'sec': 7.219, 'fnspid_val': 0.623, 'reddit_val': 0.567}
    {'epoch': 2, 'sec': 4.144, 'fnspid_val': 0.645, 'reddit_val': 0.581}
    {'epoch': 3, 'sec': 5.164, 'fnspid_val': 0.668, 'reddit_val': 0.59}
    {'epoch': 4, 'sec': 5.076, 'fnspid_val': 0.627, 'reddit_val': 0.546}
    {'epoch': 5, 'sec': 4.21, 'fnspid_val': 0.649, 'reddit_val': 0.547}
    {'epoch': 6, 'sec': 5.674, 'fnspid_val': 0.653, 'reddit_val': 0.563}
    {'epoch': 7, 'sec': 4.595, 'fnspid_val': 0.642, 'reddit_val': 0.553}
    {'epoch': 8, 'sec': 4.122, 'fnspid_val': 0.639, 'reddit_val': 0.544}
    {'epoch': 9, 'sec': 5.628, 'fnspid_val': 0.632, 'reddit_val': 0.529}
    {'epoch': 10, 'sec': 4.438, 'fnspid_val': 0.627, 'reddit_val': 0.552}
    {'epoch': 11, 'sec': 4.092, 'fnspid_val': 0.632, 'reddit_val': 0.536}
    {'epoch': 12, 'sec': 5.481, 'fnspid_val': 0.642, 'reddit_val': 0.535}
fold: train FNSPID < 2020 (+Twitter +Reddi

## 5. Final model: all FNSPID 2009–2021 (+ Twitter, + Reddit train/val), 3 seeds, fixed epochs

In [7]:
SEEDS = [42, 43, 44]
final_idx = np.where(((src == "fnspid") & (yr <= 2021)) | (src == "twitter") | ((src == "reddit") & np.isin(rs, ["train", "val"])))[0]
print(f"final training set: {len(final_idx):,} texts, {BEST_EPOCHS} epochs")
models = []
for s in SEEDS:
    t0 = time.time(); m, _ = train(final_idx, BEST_EPOCHS, s); models.append(m); print(f"  seed {s} done in {time.time()-t0:.0f}s")

def ens_proba(texts, ms=None, voc=None):
    X, ln = encode(list(texts), voc)
    return np.mean([predict_proba(m, X, ln) for m in (ms or models)], axis=0)

final training set: 21,802 texts, 3 epochs
  seed 42 done in 20s
  seed 43 done in 19s
  seed 44 done in 20s


## 6. Evaluation (v2 vs v1)
v1 = the 03b models (loaded from `bilstm_sentiment.pt`), scored on exactly the same texts.

In [8]:
ck = torch.load(os.path.join(DATA_DIR, "bilstm_sentiment.pt"), map_location="cpu")
v1_itos = ck["itos"]; v1_stoi = {w: i for i, w in enumerate(v1_itos)}
v1_models = []
for st in ck["states"]:
    m1 = BiLSTMClf(np.zeros((len(v1_itos), EMB), dtype=np.float32)).to(DEVICE); m1.load_state_dict(st); v1_models.append(m1)

def metrics(y, p): return {"accuracy": accuracy_score(y, p), "macro_F1": f1_score(y, p, average="macro"), "kappa": cohen_kappa_score(y, p)}
test_sets = {}
te = L.year >= 2022
test_sets["FNSPID 2022-23 vs LLM labels"] = (L.text[te], L.llm_label[te].map(L2I).to_numpy())
test_sets["Twitter test vs HUMAN labels"] = (tw_test.text, tw_test.label_name.map(L2I).to_numpy())
if USE_REDDIT:
    rt = RL.split == "test"
    test_sets["Reddit test days vs LLM labels"] = (RL.text[rt], RL.llm_label[rt].map(L2I).to_numpy())
rows, P_cache = [], {}
for name, (texts, y) in test_sets.items():
    for ver, ms, voc in [("v1 (03b)", v1_models, v1_stoi), ("v2 (03d)", models, stoi)]:
        P = ens_proba(texts, ms, voc); P_cache[(name, ver)] = P
        rows.append({"eval_set": name, "model": ver, "n": len(y), **metrics(y, P.argmax(1))})
oos21 = CV[(CV.val_year == 2021) & (CV.epoch == BEST_EPOCHS)].fnspid_val.iloc[0]
M2 = pd.DataFrame(rows)
print(M2.round(3).to_string(index=False))
print(f"\nFNSPID 2021, out-of-sample (rolling fold 3, epoch {BEST_EPOCHS}): macro-F1 {oos21:.3f}   [v1 on 2021: 0.607]")

                      eval_set    model    n  accuracy  macro_F1  kappa
  FNSPID 2022-23 vs LLM labels v1 (03b) 1658     0.747     0.652  0.485
  FNSPID 2022-23 vs LLM labels v2 (03d) 1658     0.730     0.643  0.471
  Twitter test vs HUMAN labels v1 (03b) 2388     0.834     0.781  0.675
  Twitter test vs HUMAN labels v2 (03d) 2388     0.830     0.782  0.676
Reddit test days vs LLM labels v1 (03b)  867     0.557     0.375  0.097
Reddit test days vs LLM labels v2 (03d)  867     0.608     0.568  0.360

FNSPID 2021, out-of-sample (rolling fold 3, epoch 3): macro-F1 0.569   [v1 on 2021: 0.607]


In [9]:
# Price consistency (same-day return, bullish − bearish), v1 vs v2 vs LLM teacher
px = pd.read_csv(os.path.join(DATA_DIR, "prices_gme_amc_bb.csv"), header=[0, 1], index_col=0, skiprows=[2]); px.index = pd.to_datetime(px.index)
ret = np.log(px["Adj Close"]).diff().stack().rename("ret").reset_index(); ret.columns = ["date", "ticker", "ret"]
Pv2 = ens_proba(L.text); Pv1 = ens_proba(L.text, v1_models, v1_stoi)
L["v2_label"] = np.array(LABELS)[Pv2.argmax(1)]; L["v1_label"] = np.array(LABELS)[Pv1.argmax(1)]
L["b2_pbear"], L["b2_pneu"], L["b2_pbull"] = Pv2[:, 0], Pv2[:, 1], Pv2[:, 2]
Lr = L.assign(date=pd.to_datetime(L.date)).merge(ret, on=["date", "ticker"], how="inner")
def spread(df, col):
    b, s = df.loc[df[col] == "bullish", "ret"], df.loc[df[col] == "bearish", "ret"]; t, p = stats.ttest_ind(b, s, equal_var=False)
    return {"n_bull": len(b), "n_bear": len(s), "spread_pp": 100 * (b.mean() - s.mean()), "t": t, "p": p}
pc = []
for period, mask in [("2022-23 (test)", Lr.year >= 2022), ("2021 (in training for v2)", Lr.year == 2021)]:
    for col, name in [("v1_label", "v1 (03b)"), ("v2_label", "v2 (03d)"), ("llm_label", "LLM teacher")]:
        pc.append({"period": period, "labels": name, **spread(Lr[mask], col)})
PC2 = pd.DataFrame(pc); print(PC2.round(3).to_string(index=False))

# External Reddit check: 2,555 BB comments scored by ChatGPT and Claude
cd = os.path.join(DATA_DIR, "Comments_w_Labels")
gpt = pd.read_csv(os.path.join(cd, "wallstreetbets_BB_Nov30_2020_Dec9_2020_body_Only_w_ChatGPTLabels.csv"), header=None, names=["body", "gpt"])
cla = pd.read_csv(os.path.join(cd, "wallstreetbets_BB_Nov30_2020_Dec9_2020_body_Only_w_ClaudeLabels.csv"), header=None, names=["body", "claude"])
C = gpt.assign(claude=cla.claude.values)
C = C[pd.to_numeric(C.gpt, errors="coerce").notna() & pd.to_numeric(C.claude, errors="coerce").notna()].astype({"gpt": float, "claude": float})
for ver, ms, voc in [("v1", v1_models, v1_stoi), ("v2", models, stoi)]:
    P = ens_proba(C.body, ms, voc); C[ver] = P[:, 2] - P[:, 0]
rc = []
for a in ["v1", "v2"]:
    for b in ["gpt", "claude"]:
        rho, p = stats.spearmanr(C[a], C[b]); rc.append({"model": a, "vs": b, "spearman": rho, "p": p})
RC2 = pd.DataFrame(rc); print("\nBB comments (Nov–Dec 2020):\n", RC2.round(3).to_string(index=False))

                   period      labels  n_bull  n_bear  spread_pp     t   p
           2022-23 (test)    v1 (03b)     207     300      4.935 5.530 0.0
           2022-23 (test)    v2 (03d)     264     294      4.030 5.281 0.0
           2022-23 (test) LLM teacher     282     239      5.019 5.787 0.0
2021 (in training for v2)    v1 (03b)     472     398      8.267 4.463 0.0
2021 (in training for v2)    v2 (03d)     496     339     13.649 7.548 0.0
2021 (in training for v2) LLM teacher     450     273     17.079 8.525 0.0

BB comments (Nov–Dec 2020):
 model     vs  spearman   p
   v1    gpt     0.105 0.0
   v1 claude     0.309 0.0
   v2    gpt     0.171 0.0
   v2 claude     0.457 0.0


## 7. Save

In [10]:
M2.to_csv(os.path.join(DATA_DIR, f"bilstm_{TAG}_metrics.csv"), index=False)
PC2.to_csv(os.path.join(DATA_DIR, f"bilstm_{TAG}_price_check.csv"), index=False)
RC2.to_csv(os.path.join(DATA_DIR, f"bilstm_{TAG}_reddit_check.csv"), index=False)
L[["ticker", "date", "target_date", "headline", "Url", "llm_label", "p_bear", "p_neu", "p_bull", "v2_label", "b2_pbear", "b2_pneu", "b2_pbull"]] \
    .to_csv(os.path.join(DATA_DIR, f"bilstm_{TAG}_scores_fnspid.csv"), index=False)
torch.save({"states": [m.state_dict() for m in models], "itos": itos, "max_len": MAX_LEN, "labels": LABELS, "epochs": BEST_EPOCHS,
            "used_reddit_labels": USE_REDDIT}, os.path.join(DATA_DIR, f"bilstm_{TAG}_sentiment.pt"))

RESCORE_REDDIT = True        # ~10 min on CPU for 913k comments
if RESCORE_REDDIT:
    t0 = time.time(); P_r = np.zeros((len(R), 3), dtype=np.float32)
    for i in range(0, len(R), 50_000):
        P_r[i:i+50_000] = ens_proba(R.body.iloc[i:i+50_000]); print(f"  {min(i+50_000, len(R)):,}/{len(R):,} ({time.time()-t0:.0f}s)")
    out = R[["ticker", "id", "created_utc", "score"]].assign(p_bear=P_r[:, 0], p_neu=P_r[:, 1], p_bull=P_r[:, 2])
    out.to_csv(os.path.join(DATA_DIR, f"bilstm_{TAG}_scores_reddit.csv"), index=False)
    print("Reddit label mix v2:", pd.Series(P_r.argmax(1)).map(dict(enumerate(LABELS))).value_counts(normalize=True).round(3).to_dict())
print("saved:", sorted(f for f in os.listdir(DATA_DIR) if f.startswith(f"bilstm_{TAG}")))

  50,000/912,984 (3s)
  100,000/912,984 (5s)
  150,000/912,984 (7s)
  200,000/912,984 (9s)
  250,000/912,984 (11s)
  300,000/912,984 (13s)
  350,000/912,984 (15s)
  400,000/912,984 (18s)
  450,000/912,984 (19s)
  500,000/912,984 (21s)
  550,000/912,984 (23s)
  600,000/912,984 (25s)
  650,000/912,984 (28s)
  700,000/912,984 (32s)
  750,000/912,984 (34s)
  800,000/912,984 (37s)
  850,000/912,984 (39s)
  900,000/912,984 (41s)
  912,984/912,984 (42s)
Reddit label mix v2: {'neutral': 0.427, 'bullish': 0.383, 'bearish': 0.19}
saved: ['bilstm_v2_reddit_cv.csv', 'bilstm_v2_reddit_metrics.csv', 'bilstm_v2_reddit_price_check.csv', 'bilstm_v2_reddit_reddit_check.csv', 'bilstm_v2_reddit_scores_fnspid.csv', 'bilstm_v2_reddit_scores_reddit.csv', 'bilstm_v2_reddit_sentiment.pt']
